In [4]:
import pandas as pd
import duckdb


In [6]:
trades_df = pd.read_csv(r"C:\repos\tradejournalbackend\files\zerodha-tradebook-fy25.csv")
trades_df

,symbol,isin,trade_date,exchange,segment,series,trade_type,auction,quantity,price,trade_id,order_id,order_execution_time,expiry_date
0,NIFTY2441022400PE,NaN,2024-04-01,NSE,FO,NaN,sell,False,50.0,116.00,173870211,1200000060667235,2024-04-01T12:16:26,2024-04-10
1,NIFTY2441022400PE,NaN,2024-04-01,NSE,FO,NaN,sell,False,550.0,116.00,173870213,1200000060667235,2024-04-01T12:16:26,2024-04-10
2,ACC24APRFUT,NaN,2024-04-02,NSE,FO,NaN,buy,False,300.0,2599.45,975636914,2500000075883607,2024-04-02T12:10:29,2024-04-25
3,ACC24APRFUT,NaN,2024-04-02,NSE,FO,NaN,buy,False,600.0,2599.50,975636918,2500000075883607,2024-04-02T12:10:29,2024-04-25
4,ACC24APRFUT,NaN,2024-04-02,NSE,FO,NaN,buy,False,300.0,2599.50,975636916,2500000075883607,2024-04-02T12:10:29,2024-04-25
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5219,INFY25MAR1760CE,NaN,2025-02-28,NSE,FO,NaN,sell,False,400.0,19.40,644726,2300000100196108,2025-02-28T12:03:09,2025-03-27
5220,TITAN25MAR3200CE,NaN,2025-02-28,NSE,FO,NaN,sell,False,175.0,41.80,730342,2700000115264965,2025-02-28T12:03:53,2025-03-27
5221,LT25MAR3300CE,NaN,2025-02-28,NSE,FO,NaN,sell,False,450.0,27.10,462068,2600000097477504,2025-02-28T12:04:16,2025-03-27
5222,RELIANCE25MAR1250CE,NaN,2025-02-28,NSE,FO,NaN,sell,False,1500.0,11.15,493124,2500000099581707,2025-02-28T12:04:35,2025-03-27


In [7]:
pnl_df = pd.read_excel(r"C:\repos\tradejournalbackend\files\zerodha-pnl-fy25.xlsx", skiprows=37)
cols=list(pnl_df.columns)
renamed_cols = list(map(lambda x: x.replace('.', '').replace('&','n').lower().replace(' ', '_'), cols))
pnl_df = pnl_df.drop(columns=['Unnamed: 0']).rename(columns=dict(zip(cols, renamed_cols)))
pnl_df

,symbol,isin,quantity,buy_value,sell_value,realized_pnl,realized_pnl_pct,previous_closing_price,open_quantity,open_quantity_type,open_value,unrealized_pnl,unrealized_pnl_pct
0,ACC24APR2500PE,NaN,1200,4.125000e+04,4.260000e+04,1350.00,3.2727,0.0,0,NaN,0.0,0.0,0.0000
1,ACC24APRFUT,NaN,1200,3.119385e+06,3.068520e+06,-50865.00,-1.6306,0.0,0,NaN,0.0,0.0,0.0000
2,ALKEM24MAY4950CE,NaN,900,1.324150e+05,2.503350e+05,117920.00,89.0534,0.0,0,NaN,0.0,0.0,0.0000
3,ALKEM24MAYFUT,NaN,900,4.602465e+06,4.378205e+06,-224260.00,-4.8726,0.0,0,NaN,0.0,0.0,0.0000
4,APOLLOTYRE24JUL470CE,NaN,5100,2.024700e+05,2.652000e+05,62730.00,30.9824,0.0,0,NaN,0.0,0.0,0.0000
...,...,...,...,...,...,...,...,...,...,...,...,...,...
531,TITAN24SEPFUT,NaN,3325,1.190682e+07,1.213426e+07,227430.00,1.9101,0.0,0,NaN,0.0,0.0,0.0000
532,TITAN25JAN3350CE,NaN,175,1.643250e+04,8.050000e+03,-8382.50,-51.0117,0.0,0,NaN,0.0,0.0,0.0000
533,TITAN25JAN3500CE,NaN,350,1.834000e+04,1.593375e+04,-2406.25,-13.1202,0.0,0,NaN,0.0,0.0,0.0000
534,TITAN25JAN3650CE,NaN,175,7.525000e+02,5.442500e+03,4690.00,623.2558,0.0,0,NaN,0.0,0.0,0.0000


In [9]:
strategy_df = pd.read_csv("strategies.csv")
strategy_df

,symbol,strategy
0,MIDCPNIFTY25JAN12300PE,manual
1,NIFTY2510923800CE,manual
2,NIFTY2510924000PE,manual
3,PETRONET25JAN340PE,manual
4,PETRONET25JANFUT,manual
...,...,...
495,HDFCBANK24APR1440PE,money in the bank
496,BANKNIFTY24APR46900PE,money in the bank
497,ICICIBANK24APR1080PE,money in the bank
498,HDFCBANK24APR1520PE,money in the bank


In [69]:
journalentry_df = duckdb.sql('''select pnl_df.symbol,
                trade_type,
                pnl_df.quantity,
                buy_value,
                sell_value,
                trade_id, entry_time, exit_time, realized_pnl, strategy
            from pnl_df inner join
                (select symbol, min(order_execution_time) as entry_time,
                        max(order_execution_time) as exit_time, min(order_id) as trade_id,
                        any_value(trade_type) as trade_type
                        from trades_df group by symbol) as T
            on pnl_df.symbol = T.symbol
            full outer join strategy_df on pnl_df.symbol = strategy_df.symbol
            where pnl_df.quantity <> 0
            order by exit_time desc
            ''').df()

journalentry_df

,symbol,trade_type,quantity,buy_value,sell_value,trade_id,entry_time,exit_time,realized_pnl,strategy
0,NIFTY25FEB22500PE,buy,600,45300.0,60.0,1200000027319166,2025-02-24T10:05:51,2025-02-27T15:25:14,-45240.0,None
1,BANKNIFTY25FEB51200CE,buy,1200,103044.0,3360.0,1800000009373502,2025-02-03T09:39:31,2025-02-25T15:22:16,-99684.0,None
2,BANKNIFTY25FEB49400PE,sell,900,589203.0,527004.0,1800000008854395,2025-02-03T09:38:03,2025-02-25T15:20:01,-62199.0,None
3,BANKNIFTY25FEB47600PE,buy,900,160551.0,48945.0,1800000009472804,2025-02-03T09:40:13,2025-02-25T15:20:00,-111606.0,None
4,BANKNIFTY25FEB48500CE,sell,600,184530.0,222021.0,1700000010783865,2025-02-24T09:27:02,2025-02-25T15:19:50,37491.0,None
...,...,...,...,...,...,...,...,...,...,...
512,ACC24APRFUT,buy,1200,3119385.0,3068520.0,2500000061106271,2024-04-02T12:10:29,2024-04-09T11:19:15,-50865.0,kumo breakout
513,HDFCBANK24APR1440PE,sell,1100,2475.0,11000.0,2400000025716040,2024-04-03T10:07:02,2024-04-09T11:11:59,8525.0,money in the bank
514,AXISBANK24APR1040PE,sell,1250,6750.0,20250.0,2500000026752880,2024-04-03T10:07:59,2024-04-09T11:10:53,13500.0,money in the bank
515,ICICIBANK24APR1050PE,sell,1400,4060.0,14000.0,2300000023127585,2024-04-03T10:07:12,2024-04-09T11:10:12,9940.0,money in the bank


In [11]:
journalentry_df[['symbol']].to_csv("strategies.csv", index=False)

In [12]:
journalentry_df['exit_month'] = journalentry_df['exit_time'].apply(lambda x: x[:7])
grp = journalentry_df.groupby(['strategy', 'exit_month'])
grp.groups.keys()

dict_keys([('blacksmith', '2024-04'), ('blacksmith', '2024-05'), ('blacksmith', '2024-06'), ('blacksmith', '2024-07'), ('blacksmith', '2024-08'), ('blacksmith', '2024-09'), ('blacksmith', '2024-10'), ('blacksmith', '2024-11'), ('blacksmith', '2024-12'), ('blacksmith', '2025-01'), ('bn 30 points', '2024-06'), ('covered call', '2024-04'), ('covered call', '2024-05'), ('covered call', '2024-06'), ('covered call', '2024-07'), ('covered call', '2024-08'), ('covered call', '2024-09'), ('covered call', '2024-10'), ('covered call', '2024-11'), ('covered call', '2024-12'), ('covered call', '2025-01'), ('heikenashi', '2024-06'), ('heikenashi', '2024-07'), ('heikenashi', '2024-08'), ('ice-cream bucket', '2024-09'), ('ice-cream bucket', '2024-10'), ('ice-cream bucket', '2024-12'), ('ice-cream bucket', '2025-01'), ('iron condor', '2024-08'), ('iron condor', '2024-09'), ('iron condor', '2024-10'), ('iron condor', '2024-11'), ('iron condor', '2024-12'), ('iron condor', '2025-01'), ('kumo breakout', '

In [131]:
grp.get_group(('blacksmith', '2025-02'))

,symbol,trade_type,quantity,buy_value,sell_value,trade_id,entry_time,exit_time,realized_pnl,strategy,exit_month
1,BANKNIFTY25FEB51200CE,buy,1200,103044.0,3360.0,1800000009373502,2025-02-03T09:39:31,2025-02-25T15:22:16,-99684.0,blacksmith,2025-02
2,BANKNIFTY25FEB49400PE,sell,900,589203.0,527004.0,1800000008854395,2025-02-03T09:38:03,2025-02-25T15:20:01,-62199.0,blacksmith,2025-02
3,BANKNIFTY25FEB47600PE,buy,900,160551.0,48945.0,1800000009472804,2025-02-03T09:40:13,2025-02-25T15:20:00,-111606.0,blacksmith,2025-02
4,BANKNIFTY25FEB48500CE,sell,600,184530.0,222021.0,1700000010783865,2025-02-24T09:27:02,2025-02-25T15:19:50,37491.0,blacksmith,2025-02
5,BANKNIFTY25FEB49400CE,sell,600,10200.0,426480.0,1800000008854228,2025-02-03T09:38:05,2025-02-25T15:19:29,416280.0,blacksmith,2025-02


In [41]:
open_trades = duckdb.sql('''select pnl_df.symbol, 
            open_quantity as quantity, 
            (case when open_quantity_type = 'buy' then open_value else 0 end) as buy_value,
            (case when open_quantity_type = 'sell' then open_value else 0 end) as sell_value,
            trade_id, entry_time,
            strftime(NOW(),'%Y-%m-%dT%H:%M:%S') as exit_time, 0 as realized_pnl,
            open_quantity_type as trade_type
            from pnl_df inner join 
                (select symbol, min(order_execution_time) as entry_time,
                        min(order_id) as trade_id 
                        from trades_df group by symbol) as T
            on pnl_df.symbol = T.symbol where pnl_df.quantity = 0''').df()

open_trades
all_df = pd.concat([open_trades,journalentry_df])

In [2]:
import deltalake
strategies_df = deltalake.DeltaTable(r"C:\repos\tradejournalbackend\deltafiles\strategies.delta").to_pandas()
strategies_df

,symbol,strategy
0,NIFTY25FEB22500PE,ice-cream bucket
1,BANKNIFTY25FEB51200CE,blacksmith
2,BANKNIFTY25FEB49400PE,blacksmith
3,BANKNIFTY25FEB47600PE,blacksmith
4,BANKNIFTY25FEB48500CE,blacksmith
...,...,...
521,HDFCBANK24APR1440PE,money in the bank
522,BANKNIFTY24APR46900PE,money in the bank
523,ICICIBANK24APR1080PE,money in the bank
524,HDFCBANK24APR1520PE,money in the bank


In [68]:
journalentry_df[journalentry_df['strategy'].isnull()][['symbol','strategy']].to_csv('strategyblank.csv', index=False)
#journalentry_df